# 13 Q1 ฉบับนักสถิติ: ถ้าไม่มี COVID ผู้โดยสารควรเป็นเท่าไร

Q1 เดิม (notebook 04) วัดการฟื้นตัวโดยเทียบปี 2023 กับปี 2019 ตรงๆ วิธีนี้สมมติว่าถ้าไม่มี COVID ผู้โดยสารจะเท่าปี 2019 ไปตลอด
แต่ก่อน COVID ผู้โดยสารไม่ได้นิ่ง เพิ่มขึ้นจนถึงปี 2016 แล้วค่อยๆ ลดลง

notebook นี้สร้างเส้น counterfactual คือ "ผู้โดยสารที่ควรจะเป็นถ้าไม่มี COVID" ด้วยโมเดล SARIMA ที่เทรนจากข้อมูลก่อน COVID เท่านั้น แล้ววัดการฟื้นตัวเทียบกับเส้นนั้น

ขั้นตอนแบบ Box-Jenkins
1. เตรียมข้อมูลและเลือกการแปลง (transformation)
2. แยกองค์ประกอบ (STL) และทดสอบ stationarity (ADF, KPSS) เพื่อหา d และ D
3. ดู ACF/PACF เพื่อระบุโมเดลที่น่าจะเหมาะ
4. ค้นหาโมเดลทุกแบบที่เป็นไปได้ เทียบด้วย AICc
5. ทดสอบความแม่นบนข้อมูลที่โมเดลไม่เคยเห็น และ rolling-origin cross-validation เทียบกับ benchmark
6. ตรวจพารามิเตอร์ทุกตัว (นัยสำคัญ, stationarity, invertibility) และ assumption ของ residual ทุกข้อ
7. เขียนสมการของโมเดล
8. พยากรณ์ มี.ค. 2020 ถึง เม.ย. 2023 แบบไม่มี COVID แล้ววัดการฟื้นตัว

ตัวแปรที่ศึกษา: ผู้โดยสารเฉลี่ยต่อวันทำงานปกติ รายเดือน (ทั้งระบบ) ช่วงเดียวกับตัวเลข 37% ใน Q1

In [1]:
import sys
import warnings
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import itertools
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, adfuller, kpss, pacf

from bart import viz
from bart.config import TAB, create_views, duck
from bart.perf import stage

viz.notebook_mode()


def test_p(test_result: Any) -> float:
    # scipy test results unpack as (statistic, p-value)
    return float(test_result[1])


def flat_line(fig, y, x0, x1, color, dash, row, col):
    # horizontal reference line inside one subplot, drawn as a trace so it works with any subplot grid
    fig.add_trace(go.Scatter(x=[x0, x1], y=[y, y], mode="lines", line=dict(color=color, dash=dash, width=1),
                             hoverinfo="skip", showlegend=False), row=row, col=col)
warnings.filterwarnings("ignore")

con = duck()
create_views(con)

## 1. ข้อมูล

ผู้โดยสารเฉลี่ยต่อวัน ของวันทำงานปกติ (ไม่รวมวันหยุด, วัน strike และวันที่ข้อมูลไม่ครบ) รายเดือน
- ก่อน COVID: ม.ค. 2011 ถึง ก.พ. 2020 (110 เดือน) ใช้สร้างโมเดล
- หลัง COVID: มี.ค. 2020 ถึง เม.ย. 2023 (38 เดือน) ใช้เทียบกับ counterfactual
- ตัด พ.ค. 2023 ออก เพราะมีข้อมูลแค่ 2 วัน

In [ ]:
monthly = con.sql("""
    SELECT month, avg(day_trips) AS trips, count(*) AS workdays
    FROM calendar
    WHERE day_status = 'normal' AND is_workday AND month <= DATE '2023-04-01'
    GROUP BY month
    ORDER BY month
""").df()
monthly["month"] = pd.to_datetime(monthly["month"])
monthly = monthly.set_index("month").asfreq("MS")

trips = pd.Series(monthly["trips"].to_numpy(dtype=float), index=monthly.index)
workdays = pd.Series(monthly["workdays"].to_numpy(dtype=float), index=monthly.index)

PRE_END = "2020-02"
POST_START = "2020-03"
pre = trips.loc[:PRE_END]
post = trips.loc[POST_START:]
log_pre = pd.Series(np.log(pre.to_numpy()), index=pre.index)

print(f"pre-COVID : {pre.index[0]:%b %Y} - {pre.index[-1]:%b %Y}, {len(pre)} months")
print(f"post-COVID: {post.index[0]:%b %Y} - {post.index[-1]:%b %Y}, {len(post)} months")
print(f"missing months: {int(trips.isna().sum())}")
yearly = pre.groupby(pre.index.year).mean().round(0).to_frame("mean trips per workday") # type: ignore
yearly

pre-COVID : Jan 2011 - Feb 2020, 110 months
post-COVID: Mar 2020 - Apr 2023, 38 months
missing months: 0


,mean trips per workday
month,
2011,355587.0
2012,380672.0
2013,394569.0
2014,411307.0
2015,427953.0
2016,432152.0
2017,417895.0
2018,412165.0
2019,409231.0


ข้อมูลครบทุกเดือน ไม่มีเดือนหาย

ผู้โดยสารเฉลี่ยต่อวันทำงาน
- เพิ่มจาก 355,587 ในปี 2011 ขึ้นไปสูงสุด 432,152 ในปี 2016
- จากนั้นลดลงเหลือ 409,231 ในปี 2019 ต่ำกว่าจุดสูงสุด 5%

ดังนั้นการใช้ "ระดับปี 2019" เป็นเส้นอ้างอิงจึงซ่อนสมมติฐานไว้ว่าการลดลงหยุดที่ปี 2019 พอดี โมเดลจะช่วยตอบว่าสมมติฐานนี้สมเหตุสมผลแค่ไหน

## 2. การแปลงข้อมูล

ใช้ log ด้วยเหตุผล 3 ข้อ
- ฤดูกาลของผู้โดยสารเป็นแบบสัดส่วน (เช่น ธันวาคมลดลงเป็น % ของระดับปกติ) log ทำให้กลายเป็นแบบบวก ซึ่ง SARIMA ต้องการ
- ค่าพยากรณ์ที่แปลงกลับด้วย exp เป็นบวกเสมอ
- ผลต่างใน log คือ % เปลี่ยนแปลง อ่านง่าย

ตรวจว่าความแปรปรวนขึ้นกับระดับไหม (แนวคิดเดียวกับวิธี Guerrero): ถ้าส่วนเบี่ยงเบนมาตรฐานรายปีโตตามค่าเฉลี่ยรายปี แปลว่าต้องแปลงแรงขึ้น

In [3]:
full_years = pre[:"2019-12"]
by_year = full_years.groupby(full_years.index.year).agg(["mean", "std"])
level_corr = float(np.corrcoef(by_year["mean"], by_year["std"])[0, 1])
cv_level = by_year["std"] / by_year["mean"]
print(f"correlation between yearly mean and yearly std: {level_corr:.2f}")
print(f"coefficient of variation by year: {cv_level.min():.3f} - {cv_level.max():.3f}")

correlation between yearly mean and yearly std: 0.22
coefficient of variation by year: 0.028 - 0.044


ค่าเฉลี่ยรายปีกับส่วนเบี่ยงเบนมาตรฐานรายปีมี correlation แค่ 0.22 ซึ่งอ่อน
coefficient of variation รายปีอยู่ในช่วงแคบ คือ 2.8% ถึง 4.4%

แปลว่าความแปรปรวนไม่ได้โตตามระดับมากนัก log จึงพอ ไม่จำเป็นต้องใช้การแปลงที่แรงกว่า

## 3. แยกองค์ประกอบด้วย STL

STL (Seasonal-Trend decomposition using Loess) แยก log ของผู้โดยสารเป็น แนวโน้ม + ฤดูกาล + ส่วนที่เหลือ
และวัดความแรงของแนวโน้มและฤดูกาล (Hyndman & Athanasopoulos): ค่าใกล้ 1 แปลว่าองค์ประกอบนั้นชัด

In [4]:
stl = STL(log_pre, period=12, robust=True).fit()
remainder_var = float(np.var(stl.resid))
trend_strength = max(0.0, 1 - remainder_var / float(np.var(stl.trend + stl.resid)))
seasonal_strength = max(0.0, 1 - remainder_var / float(np.var(stl.seasonal + stl.resid)))
print(f"trend strength   : {trend_strength:.2f}")
print(f"seasonal strength: {seasonal_strength:.2f}")

seasonal_values = np.exp(np.asarray(stl.seasonal, dtype=float))
month_totals = {}
# months since 1970-01, so the remainder of 12 gives the calendar month
month_numbers = log_pre.index.to_numpy().astype("datetime64[M]").astype(int) % 12 + 1
for position in range(len(log_pre)):
    month_number = int(month_numbers[position])
    month_totals.setdefault(month_number, []).append(float(seasonal_values[position]))
print("seasonal index by month (1.00 = average):")
for month_number in range(1, 13):
    print(f"{month_number:>2}  {np.mean(month_totals[month_number]):.3f}")

panels = [
    ("log trips", log_pre, viz.INK),
    ("trend", stl.trend, viz.ACCENT),
    ("seasonal", stl.seasonal, viz.CONTRAST),
    ("remainder", stl.resid, viz.MUTED),
]
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    subplot_titles=["Observed (log)", "Trend", "Seasonal", "Remainder"])
for row, (name, series, color) in enumerate(panels, start=1):
    if name == "remainder":
        fig.add_trace(go.Bar(x=series.index, y=series.values, marker=dict(color=color), name=name,
                             hovertemplate="%{x|%b %Y}: %{y:.4f}<extra></extra>"), row=row, col=1)
    else:
        fig.add_trace(go.Scatter(x=series.index, y=series.values, mode="lines", line=dict(color=color, width=2),
                                 name=name, hovertemplate="%{x|%b %Y}: %{y:.4f}<extra></extra>"), row=row, col=1)
fig.update_layout(showlegend=False, hovermode="x unified", margin=dict(t=130))
viz.titled(fig, f"Pre-COVID ridership: strong seasonality ({seasonal_strength:.2f}) and a trend that peaked in 2016",
           "STL decomposition of log trips per workday, Jan 2011 - Feb 2020")
viz.save(fig, "13_stl", height=820)
fig

trend strength   : 0.96
seasonal strength: 0.88
seasonal index by month (1.00 = average):
 1  0.963
 2  1.003
 3  0.999
 4  1.010
 5  1.007
 6  1.010
 7  1.000
 8  1.009
 9  1.045
10  1.033
11  1.008
12  0.917


แนวโน้มแรง 0.96 และฤดูกาลแรง 0.88 ทั้งสององค์ประกอบชัดมาก จึงต้องใช้โมเดลที่มีส่วนฤดูกาล (SARIMA) ไม่ใช่ ARIMA ธรรมดา

ฤดูกาลรายเดือน
- ธันวาคมต่ำสุด ต่ำกว่าค่าเฉลี่ย 8% (ช่วงวันหยุดยาวปลายปี)
- มกราคมต่ำกว่าค่าเฉลี่ย 4%
- กันยายนและตุลาคมสูงสุด สูงกว่าค่าเฉลี่ย 4.5% และ 3.3%

เส้นแนวโน้มขึ้นจนถึงปี 2016 แล้วค่อยๆ ลง การกลับทิศนี้จะกลับมามีผลตอนวัดความแม่นระยะยาวในส่วนที่ 7

## 4. Stationarity: หา d และ D

SARIMA ต้องการ series ที่ stationary หลัง differencing ทดสอบ 2 แบบที่มีสมมติฐานกลับกัน
- **ADF**: H0 = มี unit root (ไม่ stationary) ถ้า p < 0.05 ปฏิเสธได้ แปลว่า stationary
- **KPSS**: H0 = stationary ถ้า p > 0.05 ยอมรับได้ แปลว่า stationary

ต้องผ่านทั้งคู่ถึงจะมั่นใจ ลอง 4 แบบ: ไม่ difference, difference ปกติ (d=1), difference ฤดูกาล (D=1) และทั้งสองแบบ

In [5]:
candidates_diff = [
    ("none (d=0, D=0)", log_pre),
    ("d=1", log_pre.diff().dropna()),
    ("D=1", log_pre.diff(12).dropna()),
    ("d=1, D=1", log_pre.diff(12).diff().dropna()),
]
station_rows = []
for name, series in candidates_diff:
    adf_p = adfuller(series, autolag="AIC")[1]
    kpss_p = kpss(series, regression="c", nlags="auto")[1]
    if adf_p < 0.05 and kpss_p > 0.05:
        verdict = "stationary"
    else:
        verdict = "not stationary"
    station_rows.append({"differencing": name, "ADF p-value": round(adf_p, 3),
                         "KPSS p-value": round(kpss_p, 3), "verdict": verdict})
pd.DataFrame(station_rows).set_index("differencing")

,ADF p-value,KPSS p-value,verdict
differencing,,,
"none (d=0, D=0)",0.182,0.010,not stationary
d=1,0.432,0.091,not stationary
D=1,0.536,0.010,not stationary
"d=1, D=1",0.015,0.098,stationary


มีแค่แบบ d=1 + D=1 ที่ผ่านทั้งสองการทดสอบ (ADF p = 0.015, KPSS p = 0.098) จึงใช้ d=1 และ D=1

การ difference แบบเดียวไม่พอ
- d=1 อย่างเดียว: ADF ยังปฏิเสธ unit root ไม่ได้ (p = 0.43) เพราะฤดูกาลยังอยู่ใน series
- D=1 อย่างเดียว: KPSS ปฏิเสธ stationarity (p = 0.01) เพราะแนวโน้มยังอยู่

## 5. ACF และ PACF ของ series ที่ difference แล้ว

ใช้ ∇∇₁₂ log(y) (d=1, D=1) ดู lag 1-36 เส้นประคือขอบเขตนัยสำคัญ ±1.96/√n
วิธีอ่าน
- ACF ตัดหลัง lag 1 และ PACF ค่อยๆ ลด = MA(1) ส่วนปกติ (q=1)
- ACF มีแท่งที่ lag 12 = มีองค์ประกอบฤดูกาล ถ้า ACF ตัดหลัง 12 = seasonal MA(1) ถ้า PACF ที่ 12, 24 ลดลงช้าๆ ก็สอดคล้องกับ seasonal MA เช่นกัน

In [6]:
stationary = log_pre.diff(12).diff().dropna()
LAGS = 36
acf_values = np.asarray(acf(stationary, nlags=LAGS), dtype=float)
pacf_values = np.asarray(pacf(stationary, nlags=LAGS, method="ywm"), dtype=float)
bound = 1.96 / np.sqrt(len(stationary))

lags = list(range(1, LAGS + 1))
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["ACF", "PACF"])
for row, values in [(1, acf_values), (2, pacf_values)]:
    colors = []
    for lag in lags:
        if abs(values[lag]) > bound:
            colors.append(viz.CONTRAST)
        else:
            colors.append(viz.CONTEXT)
    fig.add_trace(go.Bar(x=lags, y=values[1:], marker=dict(color=colors), width=0.5,
                         hovertemplate="lag %{x}: %{y:.3f}<extra></extra>"), row=row, col=1)
    flat_line(fig, bound, 0.5, LAGS + 0.5, viz.MUTED, "dash", row, 1)
    flat_line(fig, -bound, 0.5, LAGS + 0.5, viz.MUTED, "dash", row, 1)
for lag in [12, 24, 36]:
    fig.add_vline(x=lag, line=dict(color=viz.GRID, width=8), opacity=0.6)
fig.update_xaxes(title_text="lag (months)", dtick=6, row=2, col=1)
fig.update_layout(showlegend=False, hovermode="closest", margin=dict(t=130))

significant_acf = []
significant_pacf = []
for lag in lags:
    if abs(acf_values[lag]) > bound:
        significant_acf.append(lag)
    if abs(pacf_values[lag]) > bound:
        significant_pacf.append(lag)
print(f"significance bound: ±{bound:.3f}")
print(f"significant ACF lags : {significant_acf}")
print(f"significant PACF lags: {significant_pacf}")
print(f"ACF lag 1 = {acf_values[1]:.3f}, lag 12 = {acf_values[12]:.3f}")

viz.titled(fig, "Spikes at lag 1 and lag 12 point to a SARIMA(0,1,1)(·,1,1)12 family",
           "ACF and PACF of the differenced log series (d=1, D=1); orange = outside the 95% bound")
viz.save(fig, "13_acf_pacf", height=620)
fig

significance bound: ±0.199
significant ACF lags : [1, 2, 11, 12]
significant PACF lags: [1, 2, 4, 11, 12, 24, 30, 34]
ACF lag 1 = -0.286, lag 12 = -0.558


ACF มีนัยสำคัญที่ lag 1 (-0.29) และ lag 12 (-0.56) ทั้งคู่ติดลบ ซึ่งเป็นรูปแบบปกติของ MA หลัง differencing
จุดเริ่มต้นจึงเป็น q=1 และ Q=1 หรือที่เรียกว่า airline model

แท่งที่ lag 11 และ 2 หลุดขอบเล็กน้อย เป็นผลข้างเคียงของ lag 1 กับ 12 เมื่อมีทั้ง MA(1) และ seasonal MA(1) จะเกิด autocorrelation ที่ lag 11 และ 13 ตามมาเอง

PACF ที่ lag 12 และ 24 มีนัยสำคัญทั้งคู่ ภาพนี้อ่านได้ทั้งแบบ seasonal MA (PACF ลดลงช้าๆ) และแบบ seasonal AR
ภาพอย่างเดียวจึงตัดสินไม่ได้ ขั้นต่อไปค้นหาทุกแบบรอบๆ จุดนี้แล้วให้ตัวเลขตัดสิน

## 6. แบ่งข้อมูลและค้นหาโมเดล

- train: ม.ค. 2011 ถึง ก.พ. 2018 (86 เดือน)
- test: มี.ค. 2018 ถึง ก.พ. 2020 (24 เดือน) โมเดลไม่เห็นช่วงนี้ตอนเลือก ใช้วัดความแม่น
- ค้นหาทุกแบบ p, q ∈ {0, 1, 2} และ P, Q ∈ {0, 1, 2} โดย d=1, D=1 (81 โมเดล) และเพิ่มกลุ่ม D=0 (ฤดูกาลแบบคงที่) อีก 36 โมเดลไว้เทียบ
- เก็บทุกตัวชี้วัด: AICc, BIC, converge ไหม, residual ผ่าน Ljung-Box / Jarque-Bera / ARCH ไหม, พารามิเตอร์มีนัยสำคัญครบไหม, รากอยู่นอกวงกลมหนึ่งหน่วยไหม และ MAPE บน test

AICc ใช้เทียบได้เฉพาะโมเดลที่ difference เท่ากัน จึงจัดอันดับแยกกลุ่ม D=1 กับ D=0

In [7]:
train = log_pre.loc[:"2018-02"]
test = log_pre.loc["2018-03":]
print(f"train {len(train)} months, test {len(test)} months")


def fit_sarima(series, order, seasonal_order, exog=None) -> Any:
    model = SARIMAX(series, exog=exog, order=order, seasonal_order=seasonal_order)
    return model.fit(disp=False, maxiter=500)


def mape(actual_log, forecast_log):
    actual = np.exp(np.asarray(actual_log))
    forecast = np.exp(np.asarray(forecast_log))
    return float(np.mean(np.abs(forecast - actual) / actual) * 100)


def diagnose(result, order, seasonal_order):
    burn = order[1] + seasonal_order[1] * seasonal_order[3]
    residuals = np.asarray(result.resid)[burn:]
    n_params = len(result.params) - 1
    ljung = acorr_ljungbox(residuals, lags=[24], model_df=n_params)["lb_pvalue"].iloc[0]
    roots = np.r_[result.arroots, result.maroots]
    if len(roots) > 0:
        min_root = float(np.min(np.abs(roots)))
    else:
        min_root = float("inf")
    p_values = result.pvalues.drop("sigma2")
    return {
        "k": n_params,
        "AICc": float(result.aicc),
        "BIC": float(result.bic),
        "converged": bool(result.mle_retvals.get("converged", True)),
        "Ljung-Box p": float(ljung),
        "Jarque-Bera p": test_p(stats.jarque_bera(residuals)),
        "ARCH p": float(het_arch(residuals, nlags=12)[1]),
        "max param p": float(p_values.max()),
        "min |root|": min_root,
    }


search_space = []
for p, q, P, Q in itertools.product(range(3), range(3), range(3), range(3)):
    search_space.append(((p, 1, q), (P, 1, Q, 12)))
for p, q, P, Q in itertools.product(range(3), range(3), range(1, 3), range(2)):
    search_space.append(((p, 1, q), (P, 0, Q, 12)))

search_rows = []
with stage("13.grid_search", models=len(search_space)):
    for order, seasonal_order in search_space:
        try:
            result = fit_sarima(train, order, seasonal_order)
        except Exception:
            continue
        row = {"model": f"SARIMA{order}{seasonal_order[:3]}12", "order": order, "seasonal": seasonal_order}
        row.update(diagnose(result, order, seasonal_order))
        forecast = result.get_forecast(len(test)).predicted_mean
        row["test MAPE %"] = mape(test, forecast)
        search_rows.append(row)

search = pd.DataFrame(search_rows)
seasonal_d = []
for seasonal_order in search["seasonal"]:
    seasonal_d.append(seasonal_order[1])
search["D"] = seasonal_d
print(f"models fitted: {len(search)}")

train 86 months, test 24 months


[perf] 13.grid_search: 518.77s | 12.99 cores | peak RAM 0.39 GB | CPU 99.3%
models fitted: 117


### 10 อันดับแรกตาม AICc (กลุ่ม D=1)

ช่อง ✓ คือผ่านเกณฑ์ที่ระดับ 5%: Ljung-Box (ไม่มี autocorrelation เหลือ), Jarque-Bera (residual ปกติ), ARCH (ความแปรปรวนคงที่), พารามิเตอร์ทุกตัวมีนัยสำคัญ และรากทุกตัวอยู่นอกวงกลมหนึ่งหน่วย (|root| > 1)

In [8]:
def flag(passed):
    if passed:
        return "✓"
    return "✗"


def summary_table(frame):
    table_rows = []
    for row in frame.to_dict("records"):
        table_rows.append({
            "model": row["model"],
            "k": row["k"],
            "AICc": round(row["AICc"], 2),
            "test MAPE %": round(row["test MAPE %"], 2),
            "converged": flag(row["converged"]),
            "Ljung-Box": flag(row["Ljung-Box p"] > 0.05),
            "Jarque-Bera": flag(row["Jarque-Bera p"] > 0.05),
            "ARCH": flag(row["ARCH p"] > 0.05),
            "params sig.": flag(row["max param p"] < 0.05),
            "roots > 1": flag(row["min |root|"] > 1.0),
        })
    return pd.DataFrame(table_rows).set_index("model")


seasonal_diff = search.loc[search["D"] == 1].sort_values(by="AICc")
summary_table(seasonal_diff.head(10))

,k,AICc,test MAPE %,converged,Ljung-Box,Jarque-Bera,ARCH,params sig.,roots > 1
model,,,,,,,,,
"SARIMA(2, 1, 0)(1, 1, 1)12",4,-384.53,1.38,✗,✓,✗,✗,✗,✓
"SARIMA(2, 1, 0)(0, 1, 2)12",4,-384.46,1.30,✓,✓,✗,✗,✗,✓
"SARIMA(2, 1, 0)(2, 1, 0)12",4,-384.28,1.96,✓,✓,✗,✗,✗,✓
"SARIMA(0, 1, 1)(1, 1, 1)12",3,-383.99,1.41,✓,✓,✓,✓,✗,✓
"SARIMA(0, 1, 1)(0, 1, 2)12",3,-383.93,1.32,✓,✓,✗,✓,✗,✓
"SARIMA(0, 1, 1)(2, 1, 0)12",3,-383.55,2.06,✓,✓,✓,✓,✗,✓
"SARIMA(2, 1, 0)(1, 1, 0)12",3,-383.17,3.28,✓,✓,✓,✗,✓,✓
"SARIMA(2, 1, 1)(1, 1, 1)12",5,-382.88,1.34,✓,✓,✓,✗,✗,✓
"SARIMA(2, 1, 1)(2, 1, 0)12",5,-382.75,2.04,✓,✓,✓,✗,✗,✓


In [9]:
no_seasonal_diff = search.loc[search["D"] == 0].sort_values(by="AICc")
summary_table(no_seasonal_diff.head(5))

,k,AICc,test MAPE %,converged,Ljung-Box,Jarque-Bera,ARCH,params sig.,roots > 1
model,,,,,,,,,
"SARIMA(2, 1, 0)(2, 0, 1)12",5,-432.48,1.62,✓,✓,✓,✓,✗,✓
"SARIMA(2, 1, 0)(2, 0, 0)12",4,-432.26,3.25,✓,✓,✓,✗,✓,✓
"SARIMA(0, 1, 2)(2, 0, 1)12",5,-430.99,1.42,✓,✓,✓,✓,✗,✓
"SARIMA(0, 1, 1)(2, 0, 1)12",4,-430.90,1.64,✓,✓,✗,✗,✗,✓
"SARIMA(0, 1, 1)(2, 0, 0)12",3,-430.60,3.24,✓,✓,✓,✗,✓,✓


อ่านผล
- AICc ของ 10 อันดับแรกต่างกันไม่ถึง 2 (-384.5 ถึง -382.7) ตามเกณฑ์ทั่วไปถือว่าข้อมูลแยกโมเดลเหล่านี้ไม่ออกด้วย AICc อย่างเดียว
- หลายตัวไม่ผ่านการตรวจ
  - อันดับ 1 ไม่ converge
  - หลายตัวมีพารามิเตอร์ที่ไม่มีนัยสำคัญ ซึ่งเป็นสัญญาณว่าโมเดลซับซ้อนเกินจำเป็น
  - บางตัว residual ไม่ปกติ (Jarque-Bera) หรือมี ARCH
- test MAPE ของกลุ่มนี้อยู่ที่ 1.3-3.3% แม่นมากสำหรับการพยากรณ์ 2 ปี
- กลุ่ม D=0 มี AICc ต่ำกว่า แต่เทียบตรงๆ ไม่ได้ เพราะ difference ต่างกัน likelihood จึงคำนวณบนข้อมูลคนละชุด
  - test MAPE ของกลุ่มนี้ (1.42-3.25%) ไม่ได้ดีกว่ากลุ่ม D=1
  - การทดสอบ stationarity ในส่วนที่ 4 บอกแล้วว่าต้องใช้ D=1

เพราะ AICc แยกไม่ออก ส่วนถัดไปจึงใช้ความแม่นนอกตัวอย่างหลายจุดเริ่มเป็นตัวตัดสิน

## 7. Rolling-origin cross-validation

test 24 เดือนเป็นแค่จุดเริ่มต้นเดียว จึงทดสอบซ้ำแบบ rolling-origin: เทรนถึงเดือน t แล้วพยากรณ์ 1-24 เดือนข้างหน้า เลื่อน t ทีละเดือนตั้งแต่ ก.พ. 2015 ถึง ก.พ. 2018 (37 จุดเริ่ม) ทุกจุดเริ่มเทรนใหม่ด้วยข้อมูลก่อนหน้าเท่านั้น
ใช้ horizon ยาวถึง 24 เดือน เพราะ counterfactual ต้องพยากรณ์ไกล 38 เดือน ความแม่นระยะยาวจึงสำคัญกว่าระยะสั้น

เทียบ
- 5 โมเดล SARIMA (d=1, D=1) ที่ AICc ดีที่สุดและผ่านการตรวจเบื้องต้น
- SARIMA(1,0,0)(0,1,1)12 แบบไม่ difference ปกติ ซึ่งพยากรณ์ระยะยาวจะค่อยๆ แบนลง (ใช้เป็นทางเลือกเปรียบเทียบ)
- benchmark: seasonal naive (ใช้ค่าเดือนเดียวกันของปีล่าสุด) และ ETS (Holt-Winters แบบ damped trend + ฤดูกาลแบบคูณ)

In [10]:
SHORTLIST = [
    ((0, 1, 1), (1, 1, 1, 12)),
    ((0, 1, 1), (0, 1, 1, 12)),
    ((0, 1, 1), (0, 1, 2, 12)),
    ((0, 1, 1), (1, 1, 0, 12)),
    ((2, 1, 0), (0, 1, 2, 12)),
]
ALTERNATIVE = ((1, 0, 0), (0, 1, 1, 12))
HORIZON = 24
origins = pd.date_range("2015-02-01", "2018-02-01", freq="MS")
NAIVE = "seasonal naive"
ETS = "ETS (damped, mult.)"


def label(order, seasonal_order):
    return f"SARIMA{order}{seasonal_order[:3]}12"


cv_models = []
for order, seasonal_order in SHORTLIST:
    cv_models.append((order, seasonal_order))
cv_models.append(ALTERNATIVE)

cv_errors = {}
for order, seasonal_order in cv_models:
    cv_errors[label(order, seasonal_order)] = []
cv_errors[NAIVE] = []
cv_errors[ETS] = []

with stage("13.rolling_cv", origins=len(origins), models=len(cv_models) + 2):
    for origin in origins:
        history = log_pre.loc[:origin]
        future = log_pre.loc[origin + pd.offsets.MonthBegin(1):].iloc[:HORIZON]
        actual = np.exp(future.to_numpy())
        h = len(future)
        for order, seasonal_order in cv_models:
            result = fit_sarima(history, order, seasonal_order)
            forecast = np.exp(np.asarray(result.get_forecast(h).predicted_mean))
            cv_errors[label(order, seasonal_order)].append(np.abs(forecast - actual) / actual * 100)
        last_year = np.exp(history.to_numpy()[-12:])
        naive = last_year[np.arange(h) % 12]
        cv_errors[NAIVE].append(np.abs(naive - actual) / actual * 100)
        ets = ExponentialSmoothing(pd.Series(np.exp(history.to_numpy()), index=history.index), trend="add", damped_trend=True,
                                   seasonal="mul", seasonal_periods=12).fit()
        ets_forecast = np.asarray(ets.forecast(h))
        cv_errors[ETS].append(np.abs(ets_forecast - actual) / actual * 100)

cv_by_horizon = {}
for name, error_list in cv_errors.items():
    per_h = []
    for step in range(HORIZON):
        values = []
        for errors in error_list:
            if step < len(errors):
                values.append(errors[step])
        per_h.append(float(np.mean(values)))
    cv_by_horizon[name] = per_h

STEPS = list(range(1, HORIZON + 1))
cv_table = pd.DataFrame(cv_by_horizon, index=STEPS).T
cv_table["mean h1-12"] = cv_table[list(range(1, 13))].mean(axis=1)
cv_table["mean h1-24"] = cv_table[STEPS].mean(axis=1)
cv_table = cv_table.sort_values("mean h1-24")
cv_table[[1, 3, 6, 12, 18, 24, "mean h1-12", "mean h1-24"]].round(2)

[perf] 13.rolling_cv: 131.76s | 17.99 cores | peak RAM 0.384 GB | CPU 93.2%


,1,3,6,12,18,24,mean h1-12,mean h1-24
"SARIMA(1, 0, 0)(0, 1, 1)12",1.04,1.40,1.57,2.56,2.98,3.86,1.80,2.40
seasonal naive,2.84,2.59,2.31,2.24,3.14,3.30,2.44,2.86
"ETS (damped, mult.)",1.13,1.45,2.01,4.03,5.86,7.48,2.39,4.12
"SARIMA(0, 1, 1)(1, 1, 0)12",1.38,1.77,2.64,4.48,6.43,7.64,2.89,4.63
"SARIMA(0, 1, 1)(0, 1, 2)12",1.36,1.87,2.75,5.18,7.43,8.99,3.12,5.25
"SARIMA(2, 1, 0)(0, 1, 2)12",1.35,1.83,2.75,5.25,7.49,9.05,3.13,5.28
"SARIMA(0, 1, 1)(1, 1, 1)12",1.38,1.81,2.90,5.25,7.49,9.23,3.21,5.36
"SARIMA(0, 1, 1)(0, 1, 1)12",1.19,1.76,2.78,5.51,8.58,11.34,3.13,5.94


In [11]:
main_family = []
for order, seasonal_order in SHORTLIST:
    main_family.append(label(order, seasonal_order))
best_short = cv_table.loc[main_family, 1].idxmin()
wins = 0
for step in STEPS:
    if cv_table.loc[main_family, step].min() < cv_table.loc[NAIVE, step]:
        wins += 1
    else:
        break
print(f"months ahead where the best d=1 SARIMA beats seasonal naive (from h=1): {wins}")

CV_COLORS = {}
palette = [viz.SERIES[2], viz.SERIES[3], viz.SERIES[4], viz.ACCENT, viz.SERIES[6], viz.SERIES[1]]
for position, (order, seasonal_order) in enumerate(cv_models):
    CV_COLORS[label(order, seasonal_order)] = palette[position]
fig = go.Figure()
for name in cv_table.index:
    if name == NAIVE:
        color = viz.INK
        dash = "dot"
    elif name == ETS:
        color = viz.MUTED
        dash = "dot"
    else:
        color = CV_COLORS[name]
        dash = "solid"
    fig.add_trace(go.Scatter(
        x=STEPS, y=cv_table.loc[name, STEPS].values, mode="lines+markers", name=name,
        line=dict(color=color, width=2.5, dash=dash), marker=dict(size=6),
        hovertemplate="h=%{x}: MAPE %{y:.2f}%<extra>" + name + "</extra>",
    ))
fig.add_vrect(x0=0.5, x1=wins + 0.5, fillcolor=viz.ACCENT, opacity=0.06, line_width=0)
fig.add_annotation(x=(wins + 1) / 2, y=1, yref="paper", text=f"d=1 SARIMA beats<br>seasonal naive<br>h = 1-{wins}", showarrow=False,
                   yanchor="top", font=dict(size=12, color=viz.ACCENT))
fig.update_xaxes(title_text="months ahead", dtick=2)
fig.update_yaxes(title_text="MAPE across 37 forecast origins", ticksuffix="%", rangemode="tozero")
fig.update_layout(hovermode="x unified", legend=dict(y=1.16), margin=dict(t=190))
if wins >= HORIZON:
    cv_title = "SARIMA beats the seasonal naive benchmark at every horizon up to 24 months"
else:
    cv_title = f"SARIMA (d=1) beats the seasonal naive benchmark only for the first {wins} months"
viz.titled(fig, cv_title, "Rolling-origin cross-validation, 37 origins Feb 2015 - Feb 2018; error grows with the horizon "
           "because d=1 models extend the recent trend, and the trend turned in 2016")
viz.save(fig, "13_cv_horizon", width=1200, height=640)
fig

months ahead where the best d=1 SARIMA beats seasonal naive (from h=1): 4


ข้อค้นพบที่ต้องรายงานตรงๆ
- 1-4 เดือนข้างหน้า SARIMA แม่นมาก error เฉลี่ยราว 1.2-2.3% ดีกว่า seasonal naive (2.5-2.8%)
- ยิ่งพยากรณ์ไกล error ของ SARIMA กลุ่ม d=1 ยิ่งโต
  - ที่ 24 เดือน โมเดลที่ดีที่สุดในกลุ่มนี้คลาด 7.6%
  - seasonal naive คลาดแค่ 3.3%
- สาเหตุคือโมเดล d=1 ต่อแนวโน้มล่าสุดออกไปเป็นเส้นตรง ช่วง 2015-2018 แนวโน้มกลับทิศ (ขึ้นจนถึง 2016 แล้วลง) การต่อเส้นตรงจึงพลาด ส่วน seasonal naive สมมติว่าระดับคงที่ ซึ่งบังเอิญใกล้ความจริงช่วงนั้น
- SARIMA(1,0,0)(0,1,1)12 ที่ไม่ใช้ d ดีที่สุดทุกระยะ แต่มีปัญหาเชิงทฤษฎี (ดูส่วนที่ 8)

ความหมาย: ความแม่นระยะยาวของทุกโมเดลมีขีดจำกัดจริง ถ้ารายงาน counterfactual 38 เดือนเป็นตัวเลขเดียวแบบมั่นใจจะเกินจริง
notebook นี้จึงรายงานพร้อมช่วงพยากรณ์ และเทียบผลข้ามหลายโมเดลในส่วนที่ 14

## 8. เลือกโมเดล

กฎการเลือก (เขียนไว้ก่อนดูผล counterfactual เพื่อไม่ให้เลือกโมเดลตามคำตอบที่อยากได้)
1. อยู่ในกลุ่มที่การทดสอบ stationarity ระบุ (d=1, D=1)
2. residual บน train ต้องผ่าน Ljung-Box, Jarque-Bera และ ARCH ที่ระดับ 5% (ไม่อย่างนั้นช่วงพยากรณ์เชื่อไม่ได้)
3. พารามิเตอร์ทุกตัวต้องมีนัยสำคัญที่ 5% และรากทุกตัวต้องอยู่นอกวงกลมหนึ่งหน่วย
4. ในกลุ่มที่ผ่าน เลือก MAPE เฉลี่ย rolling CV (h 1-24) ต่ำที่สุด

SARIMA(1,0,0)(0,1,1)12 ไม่เข้ารอบตามข้อ 1 ด้วยเหตุผลในส่วนถัดไป

In [12]:
eligible = []
for order, seasonal_order in SHORTLIST:
    name = label(order, seasonal_order)
    row = search[search["model"] == name].iloc[0]
    residual_ok = row["Ljung-Box p"] > 0.05 and row["Jarque-Bera p"] > 0.05 and row["ARCH p"] > 0.05
    params_ok = row["max param p"] < 0.05 and row["min |root|"] > 1.0
    eligible.append({"model": name, "residuals pass": flag(residual_ok), "params sig. + roots": flag(params_ok),
                     "CV MAPE h1-24 %": round(cv_table.loc[name, "mean h1-24"], 3),
                     "test MAPE %": round(row["test MAPE %"], 3), "AICc": round(row["AICc"], 2),
                     "_ok": residual_ok and params_ok, "_order": order, "_seasonal": seasonal_order})
eligible_table = pd.DataFrame(eligible).sort_values("CV MAPE h1-24 %")
chosen = eligible_table[eligible_table["_ok"]].iloc[0]
ORDER = chosen["_order"]
SEASONAL = chosen["_seasonal"]
CHOSEN_NAME = chosen["model"]
print(f"chosen model: {CHOSEN_NAME}")
eligible_table.drop(columns=["_ok", "_order", "_seasonal"]).set_index("model")

chosen model: SARIMA(0, 1, 1)(1, 1, 0)12


,residuals pass,params sig. + roots,CV MAPE h1-24 %,test MAPE %,AICc
model,,,,,
"SARIMA(0, 1, 1)(1, 1, 0)12",✓,✓,4.628,3.338,-382.34
"SARIMA(0, 1, 1)(0, 1, 2)12",✗,✗,5.248,1.322,-383.93
"SARIMA(2, 1, 0)(0, 1, 2)12",✗,✗,5.283,1.305,-384.46
"SARIMA(0, 1, 1)(1, 1, 1)12",✓,✗,5.365,1.415,-383.99
"SARIMA(0, 1, 1)(0, 1, 1)12",✗,✓,5.936,2.778,-382.22


In [13]:
alternative_fit = fit_sarima(log_pre, ALTERNATIVE[0], ALTERNATIVE[1])
ar_value = float(alternative_fit.params["ar.L1"])
ar_low, ar_high = alternative_fit.conf_int().loc["ar.L1"]
print(f"{label(ALTERNATIVE[0], ALTERNATIVE[1])}: ar.L1 = {ar_value:.4f}, 95% CI [{ar_low:.4f}, {ar_high:.4f}]")
if ar_high >= 1:
    print("the interval contains 1: a unit root cannot be rejected, so this is the d=1 model in disguise")

SARIMA(1, 0, 0)(0, 1, 1)12: ar.L1 = 0.9878, 95% CI [0.9458, 1.0298]
the interval contains 1: a unit root cannot be rejected, so this is the d=1 model in disguise


### Diebold-Mariano test

ทดสอบว่าโมเดลที่เลือกแม่นต่างจาก benchmark อย่างมีนัยสำคัญทางสถิติไหม ใช้ error ยกกำลังสองที่ horizon 1, 6, 12 และ 24 และปรับแก้ Harvey-Leybourne-Newbold สำหรับตัวอย่างเล็ก
H0: ความแม่นเท่ากัน ค่า DM ติดลบ = SARIMA error น้อยกว่า

In [14]:
def diebold_mariano(errors_a, errors_b, horizon):
    loss = np.asarray(errors_a) ** 2 - np.asarray(errors_b) ** 2
    n = len(loss)
    mean_loss = float(np.mean(loss))
    centred = loss - mean_loss
    variance = float(np.sum(centred ** 2)) / n
    for lag in range(1, horizon):
        variance += 2 * float(np.sum(centred[lag:] * centred[:-lag])) / n
    if variance <= 0:
        variance = float(np.sum(centred ** 2)) / n
    statistic = mean_loss / np.sqrt(variance / n)
    correction = np.sqrt((n + 1 - 2 * horizon + horizon * (horizon - 1) / n) / n)
    statistic = statistic * correction
    p_value = 2 * (1 - stats.t.cdf(abs(statistic), df=n - 1))
    return float(statistic), float(p_value)


def dm_verdict(statistic, p_value):
    if p_value >= 0.05:
        return "no significant difference"
    if statistic < 0:
        return "SARIMA better"
    return "benchmark better"


dm_rows = []
for benchmark in [NAIVE, ETS]:
    for step in [1, 6, 12, 24]:
        errors_chosen = []
        errors_bench = []
        for index, errors in enumerate(cv_errors[CHOSEN_NAME]):
            if step - 1 < len(errors):
                errors_chosen.append(errors[step - 1])
                errors_bench.append(cv_errors[benchmark][index][step - 1])
        statistic, p_value = diebold_mariano(errors_chosen, errors_bench, step)
        dm_rows.append({"compare with": benchmark, "horizon": step, "pairs": len(errors_chosen),
                        "DM statistic": round(statistic, 2), "p-value": round(p_value, 4),
                        "verdict": dm_verdict(statistic, p_value)})
pd.DataFrame(dm_rows)

,compare with,horizon,pairs,DM statistic,p-value,verdict
0,seasonal naive,1,37,-4.40,0.0001,SARIMA better
1,seasonal naive,6,37,0.47,0.6424,no significant difference
2,seasonal naive,12,37,1.30,0.2006,no significant difference
3,seasonal naive,24,37,1.33,0.1914,no significant difference
4,"ETS (damped, mult.)",1,37,1.95,0.0584,no significant difference
5,"ETS (damped, mult.)",6,37,1.41,0.1676,no significant difference
6,"ETS (damped, mult.)",12,37,0.91,0.3713,no significant difference
7,"ETS (damped, mult.)",24,37,0.83,0.4105,no significant difference


โมเดลที่เลือกคือ SARIMA(0,1,1)(1,1,0)12 เป็นตัวเดียวใน shortlist ที่ผ่านทุกเกณฑ์ และมี CV error ต่ำสุดในกลุ่ม d=1 (4.6% เทียบกับ 5.2-5.9%)

ข้อสังเกต: บน test ชุดเดียว (มี.ค. 2018 ถึง ก.พ. 2020) โมเดลนี้คลาด 3.3% ขณะที่ตัวอื่นคลาดราว 1.3%
แต่ test ชุดเดียวคือจุดเริ่มต้นเดียว อาจโชคดีหรือโชคร้ายได้ rolling CV 37 จุดเริ่มน่าเชื่อถือกว่า

SARIMA(1,0,0)(0,1,1)12 ไม่เข้ารอบ ค่า AR ประมาณได้ 0.988 และช่วงความเชื่อมั่น 95% ครอบเลข 1 แปลว่าปฏิเสธ unit root ไม่ได้
โมเดลนี้จึงเป็นโมเดล d=1 ที่แฝงมา ตรงกับผล ADF ในส่วนที่ 4 ที่ใช้ได้ดีในระยะยาวเพราะค่าที่น้อยกว่า 1 นิดหน่อยทำให้ค่าพยากรณ์ค่อยๆ แบนลง
แต่โครงสร้างแบบนี้ไม่มีหลักฐานทางสถิติรองรับ จึงเก็บไว้เป็นทางเลือกในการทดสอบความไว

Diebold-Mariano
- ที่ 1 เดือน โมเดลที่เลือกแม่นกว่า seasonal naive อย่างมีนัยสำคัญ (p = 0.0001)
- ที่ 6, 12 และ 24 เดือน ไม่มีความต่างที่มีนัยสำคัญกับทั้ง seasonal naive และ ETS (p 0.19 ถึง 0.64)
- กับ ETS ที่ 1 เดือน ไม่ต่างอย่างมีนัยสำคัญ (p = 0.058)

แปลว่าในระยะยาว ข้อมูลก่อน COVID ไม่มากพอจะบอกได้ว่าวิธีไหนดีกว่า ซึ่งยืนยันว่าต้องทดสอบความไวข้ามโมเดล

## 9. ตรวจพารามิเตอร์ทุกตัว

fit โมเดลที่เลือกบน train แล้วดูค่าประมาณ, standard error, z, p-value และช่วงความเชื่อมั่น 95%
ตรวจเพิ่ม
- **joint Wald test** ของพารามิเตอร์ฤดูกาลทั้งหมด: พารามิเตอร์ที่ correlated กันอาจไม่มีนัยสำคัญทีละตัว แต่มีนัยสำคัญเมื่อทดสอบร่วมกัน
- **correlation ของตัวประมาณ**: ถ้าสูงมาก แปลว่าพารามิเตอร์แย่งกันอธิบาย
- **likelihood-ratio test** เทียบกับโมเดลที่ตัดพารามิเตอร์ที่ไม่มีนัยสำคัญออก

In [15]:
train_fit = fit_sarima(train, ORDER, SEASONAL)
conf = train_fit.conf_int()
param_rows = []
for name in train_fit.params.index:
    param_rows.append({
        "parameter": name,
        "estimate": round(float(train_fit.params[name]), 4),
        "std error": round(float(train_fit.bse[name]), 4),
        "z": round(float(train_fit.tvalues[name]), 2),
        "p-value": round(float(train_fit.pvalues[name]), 4),
        "95% CI low": round(float(conf.loc[name].iloc[0]), 4),
        "95% CI high": round(float(conf.loc[name].iloc[1]), 4),
    })
pd.DataFrame(param_rows).set_index("parameter")

,estimate,std error,z,p-value,95% CI low,95% CI high
parameter,,,,,,
ma.L1,-0.3718,0.1176,-3.16,0.0016,-0.6022,-0.1413
ar.S.L12,-0.6271,0.0810,-7.74,0.0000,-0.7859,-0.4684
sigma2,0.0003,0.0000,5.65,0.0000,0.0002,0.0004


In [16]:
seasonal_params = []
for name in train_fit.params.index:
    if ".S." in name:
        seasonal_params.append(name)
restriction = np.zeros((len(seasonal_params), len(train_fit.params)))
for row_index, name in enumerate(seasonal_params):
    restriction[row_index, list(train_fit.params.index).index(name)] = 1.0
wald = train_fit.wald_test(restriction, scalar=True)
print(f"joint Wald test of seasonal terms {seasonal_params}: chi2 = {float(wald.statistic):.2f}, p = {float(wald.pvalue):.5f}")

covariance = train_fit.cov_params()
names = list(train_fit.params.index.drop("sigma2"))
corr_rows = []
for a, b in itertools.combinations(names, 2):
    corr_value = float(covariance.loc[a, b] / np.sqrt(covariance.loc[a, a] * covariance.loc[b, b]))
    corr_rows.append({"pair": f"{a} vs {b}", "correlation": round(corr_value, 3)})
print(pd.DataFrame(corr_rows).to_string(index=False))

weakest = train_fit.pvalues.drop("sigma2").idxmax()
reduced_seasonal = list(SEASONAL)
reduced_order = list(ORDER)
if weakest.startswith("ma.S"):
    reduced_seasonal[2] -= 1
elif weakest.startswith("ar.S"):
    reduced_seasonal[0] -= 1
elif weakest.startswith("ma."):
    reduced_order[2] -= 1
else:
    reduced_order[0] -= 1
reduced_fit = fit_sarima(train, tuple(reduced_order), tuple(reduced_seasonal))
lr_statistic = 2 * (train_fit.llf - reduced_fit.llf)
lr_p = 1 - stats.chi2.cdf(lr_statistic, df=1)
reduced_name = label(tuple(reduced_order), tuple(reduced_seasonal))
print(f"weakest parameter: {weakest} (p = {train_fit.pvalues[weakest]:.3f})")
print(f"likelihood-ratio test vs {reduced_name}: LR = {lr_statistic:.2f}, p = {lr_p:.4f}")
full_mape = float(search.loc[search["model"] == CHOSEN_NAME, "test MAPE %"].iloc[0])
reduced_mape = float(search.loc[search["model"] == reduced_name, "test MAPE %"].iloc[0])
print(f"test MAPE: full {full_mape:.2f}% vs reduced {reduced_mape:.2f}%")

joint Wald test of seasonal terms ['ar.S.L12']: chi2 = 59.96, p = 0.00000
             pair  correlation
ma.L1 vs ar.S.L12        0.009
weakest parameter: ma.L1 (p = 0.002)
likelihood-ratio test vs SARIMA(0, 1, 0)(1, 1, 0)12: LR = 8.49, p = 0.0036
test MAPE: full 3.34% vs reduced 3.76%


พารามิเตอร์ทั้งสองตัวมีนัยสำคัญมาก และช่วงความเชื่อมั่นไม่ครอบ 0
- θ₁ (ma.L1) = -0.372, p = 0.002: ความคลาดเคลื่อนของเดือนก่อนถูกชดเชยบางส่วนในเดือนนี้
- Φ₁ (ar.S.L12) = -0.627, p < 0.001: ถ้าการเปลี่ยนแปลงรายปีของเดือนหนึ่งสูงกว่าปกติ ปีถัดไปเดือนเดียวกันมักกลับทางบางส่วน

ตรวจเพิ่ม
- correlation ระหว่างตัวประมาณทั้งสองคือ 0.009 แทบเป็นศูนย์ ไม่แย่งกันอธิบาย ค่าประมาณจึงเสถียร
- likelihood-ratio test: ถ้าตัด ma.L1 ออก โมเดลแย่ลงอย่างมีนัยสำคัญ (LR = 8.49, p = 0.004) และ test MAPE แย่ลงจาก 3.34% เป็น 3.76% จึงต้องเก็บไว้
- σ² มีค่าราว 0.0003 คือ residual sd ราว 1.7% ต่อเดือน
- joint Wald test มีพารามิเตอร์ฤดูกาลตัวเดียว จึงให้ผลเท่ากับ z-test (z² = 59.96)

## 10. ตรวจ assumption ของ residual

SARIMA ที่ใช้ได้ต้องมี residual เป็น white noise ปกติ ความแปรปรวนคงที่ ตรวจทั้งด้วย test และด้วยภาพ
- **Ljung-Box** ที่ lag 6, 12, 18, 24 (H0: ไม่มี autocorrelation)
- **Jarque-Bera** และ **Shapiro-Wilk** (H0: แจกแจงปกติ)
- **ARCH LM** (H0: ไม่มี heteroskedasticity แบบมีเงื่อนไข)
- **Breusch-Pagan แบบง่าย**: residual² กับเวลา (H0: ความแปรปรวนไม่เปลี่ยนตามเวลา)
- **t-test** ค่าเฉลี่ย residual = 0

In [17]:
burn = ORDER[1] + SEASONAL[1] * 12
residuals = pd.Series(np.asarray(train_fit.resid)[burn:], index=train.index[burn:])
standardized = residuals / residuals.std()
n_params = len(train_fit.params) - 1

assumption_rows = []
ljung = acorr_ljungbox(residuals, lags=[6, 12, 18, 24], model_df=n_params)
for lag, p_value in zip([6, 12, 18, 24], ljung["lb_pvalue"]):
    assumption_rows.append({"test": f"Ljung-Box lag {lag}", "H0": "no autocorrelation", "p-value": float(p_value)})
assumption_rows.append({"test": "Jarque-Bera", "H0": "normal", "p-value": test_p(stats.jarque_bera(residuals))})
assumption_rows.append({"test": "Shapiro-Wilk", "H0": "normal", "p-value": test_p(stats.shapiro(residuals))})
assumption_rows.append({"test": "ARCH LM (12 lags)", "H0": "constant variance", "p-value": float(het_arch(residuals, nlags=12)[1])})
time_index = np.arange(len(residuals))
squared = residuals.to_numpy() ** 2
trend_test = stats.pearsonr(time_index, squared)
assumption_rows.append({"test": "residual² vs time", "H0": "variance not trending", "p-value": test_p(trend_test)})
mean_test = stats.ttest_1samp(residuals.to_numpy(), 0.0)
assumption_rows.append({"test": "t-test mean = 0", "H0": "zero mean", "p-value": test_p(mean_test)})

assumptions = pd.DataFrame(assumption_rows)
verdicts = []
for p_value in assumptions["p-value"]:
    verdicts.append(flag(p_value > 0.05))
assumptions["pass at 5%"] = verdicts
assumptions["p-value"] = assumptions["p-value"].round(4)
print(f"residuals: n = {len(residuals)}, skewness = {stats.skew(residuals):.3f}, excess kurtosis = {stats.kurtosis(residuals):.3f}")
assumptions.set_index("test")

residuals: n = 73, skewness = -0.150, excess kurtosis = 1.050


,H0,p-value,pass at 5%
test,,,
Ljung-Box lag 6,no autocorrelation,0.3931,✓
Ljung-Box lag 12,no autocorrelation,0.2189,✓
Ljung-Box lag 18,no autocorrelation,0.5906,✓
Ljung-Box lag 24,no autocorrelation,0.5707,✓
Jarque-Bera,normal,0.1631,✓
Shapiro-Wilk,normal,0.1763,✓
ARCH LM (12 lags),constant variance,0.1280,✓
residual² vs time,variance not trending,0.2205,✓
t-test mean = 0,zero mean,0.0868,✓


In [18]:
resid_acf = np.asarray(acf(residuals, nlags=24), dtype=float)
resid_bound = 1.96 / np.sqrt(len(residuals))
theoretical, ordered = stats.probplot(standardized, dist="norm")[0]

fig = make_subplots(rows=2, cols=2, vertical_spacing=0.14, horizontal_spacing=0.08,
                    subplot_titles=["Standardized residuals over time", "Residual ACF",
                                    "Normal Q-Q plot", "Distribution vs normal curve"])
fig.add_trace(go.Scatter(x=standardized.index, y=standardized.values, mode="lines+markers",
                         line=dict(color=viz.ACCENT, width=1.5), marker=dict(size=5),
                         hovertemplate="%{x|%b %Y}: %{y:.2f}<extra></extra>"), row=1, col=1)
for level in [-2, 2]:
    flat_line(fig, level, standardized.index[0], standardized.index[-1], viz.CONTRAST, "dot", 1, 1)
acf_colors = []
for lag in range(1, 25):
    if abs(resid_acf[lag]) > resid_bound:
        acf_colors.append(viz.CONTRAST)
    else:
        acf_colors.append(viz.ACCENT)
fig.add_trace(go.Bar(x=list(range(1, 25)), y=resid_acf[1:], marker=dict(color=acf_colors), width=0.5,
                     hovertemplate="lag %{x}: %{y:.3f}<extra></extra>"), row=1, col=2)
flat_line(fig, resid_bound, 0.5, 24.5, viz.MUTED, "dash", 1, 2)
flat_line(fig, -resid_bound, 0.5, 24.5, viz.MUTED, "dash", 1, 2)
fig.add_trace(go.Scatter(x=theoretical, y=ordered, mode="markers", marker=dict(color=viz.ACCENT, size=7),
                         hovertemplate="theoretical %{x:.2f}, sample %{y:.2f}<extra></extra>"), row=2, col=1)
fig.add_trace(go.Scatter(x=[-3, 3], y=[-3, 3], mode="lines", line=dict(color=viz.CONTRAST, dash="dash"),
                         hoverinfo="skip"), row=2, col=1)
fig.add_trace(go.Histogram(x=standardized.values, histnorm="probability density", nbinsx=16,
                           marker=dict(color=viz.CONTEXT, line=dict(color="white", width=1)),
                           hovertemplate="%{x}: %{y:.2f}<extra></extra>"), row=2, col=2)
grid = np.linspace(-3.5, 3.5, 200)
fig.add_trace(go.Scatter(x=grid, y=stats.norm.pdf(grid), mode="lines", line=dict(color=viz.CONTRAST, width=2.5),
                         hoverinfo="skip"), row=2, col=2)
fig.update_xaxes(title_text="lag", row=1, col=2)
fig.update_xaxes(title_text="theoretical quantile", row=2, col=1)
fig.update_yaxes(title_text="sample quantile", row=2, col=1)
fig.update_layout(showlegend=False, hovermode="closest", margin=dict(t=130), bargap=0.05)
passed = int((assumptions["pass at 5%"] == "✓").sum())
viz.titled(fig, f"Residuals behave like white noise: {passed} of {len(assumptions)} assumption tests pass",
           f"Diagnostics of {CHOSEN_NAME} fitted on Jan 2011 - Feb 2018")
viz.save(fig, "13_residual_diagnostics", height=780)
fig

residual บน train ผ่านทั้ง 9 การทดสอบ
- ไม่มี autocorrelation เหลือ: Ljung-Box ทุก lag p > 0.2
- residual แจกแจงปกติ: Jarque-Bera p = 0.16, Shapiro-Wilk p = 0.18
- ความแปรปรวนคงที่: ARCH p = 0.13 และ residual² ไม่เพิ่มตามเวลา p = 0.22
- ค่าเฉลี่ยเป็นศูนย์: p = 0.087

ภาพยืนยันผลเดียวกัน
- Q-Q plot อยู่บนเส้นตรง ยกเว้นปลาย 2 จุด (ต.ค. 2012 และ ต.ค. 2013)
- residual ACF ไม่มีแท่งไหนหลุดขอบ
- histogram ใกล้โค้งปกติ excess kurtosis 1.05 คือหางหนากว่าปกติเล็กน้อย

### Stationarity และ invertibility

พหุนาม AR ต้อง stationary และพหุนาม MA ต้อง invertible คือรากทุกตัวต้องอยู่นอกวงกลมหนึ่งหน่วย (|root| > 1) เทียบเท่ากับ inverse root อยู่ในวงกลม

In [19]:
angle = np.linspace(0, 2 * np.pi, 400)
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.cos(angle), y=np.sin(angle), mode="lines", line=dict(color=viz.MUTED, width=1.5),
                         name="unit circle", hoverinfo="skip"))
root_groups = [("AR (inverse roots)", train_fit.arroots, viz.CONTRAST, "diamond"),
               ("MA (inverse roots)", train_fit.maroots, viz.ACCENT, "circle")]
for name, roots, color, symbol in root_groups:
    if len(roots) == 0:
        continue
    inverse = 1 / np.asarray(roots)
    fig.add_trace(go.Scatter(x=inverse.real, y=inverse.imag, mode="markers", name=name,
                             marker=dict(color=color, size=12, symbol=symbol, line=dict(color="white", width=1.5)),
                             hovertemplate="re %{x:.3f}, im %{y:.3f}<extra>" + name + "</extra>"))
fig.update_xaxes(range=[-1.2, 1.2], scaleanchor="y", scaleratio=1, zeroline=True, zerolinecolor=viz.GRID)
fig.update_yaxes(range=[-1.2, 1.2], zeroline=True, zerolinecolor=viz.GRID)
fig.update_layout(hovermode="closest")
all_roots = np.r_[train_fit.arroots, train_fit.maroots]
smallest = float(np.min(np.abs(all_roots)))
print(f"smallest |root| = {smallest:.3f} (must be > 1)")
viz.titled(fig, "Inverse roots inside the unit circle: stationary and invertible",
           f"Smallest |root| = {smallest:.3f}; seasonal roots sit on a ring because they are roots of B^12")
viz.save(fig, "13_unit_roots", width=900, height=860)
fig

smallest |root| = 1.040 (must be > 1)


## 11. ความแม่นบนข้อมูลที่ไม่เคยเห็น (มี.ค. 2018 ถึง ก.พ. 2020)

วัดหลายตัวชี้วัด
- **MAPE, MAE, RMSE** ความคลาดเคลื่อนของค่าพยากรณ์
- **MASE** เทียบกับ seasonal naive บน train (ต่ำกว่า 1 = ดีกว่า seasonal naive)
- **coverage** ของช่วงพยากรณ์ 80% และ 95%: ค่าจริงควรตกในช่วงประมาณ 80% และ 95% ตามลำดับ

In [20]:
test_forecast = train_fit.get_forecast(len(test))
test_mean = np.exp(test_forecast.predicted_mean)
test_ci95 = np.exp(test_forecast.conf_int(alpha=0.05))
test_ci80 = np.exp(test_forecast.conf_int(alpha=0.20))
actual_test = pd.Series(np.exp(test.to_numpy()), index=test.index)

errors = actual_test - test_mean
train_level = np.exp(train.to_numpy())
naive_scale = float(np.mean(np.abs(train_level[12:] - train_level[:-12])))
accuracy = {
    "MAPE %": float(np.mean(np.abs(errors) / actual_test) * 100),
    "MAE (trips per workday)": float(np.mean(np.abs(errors))),
    "RMSE (trips per workday)": float(np.sqrt(np.mean(errors ** 2))),
    "MASE": float(np.mean(np.abs(errors)) / naive_scale),
    "bias % (mean error / mean actual)": float(np.mean(errors) / np.mean(actual_test) * 100),
    "80% interval coverage %": float(np.mean((actual_test >= test_ci80.iloc[:, 0]) & (actual_test <= test_ci80.iloc[:, 1])) * 100),
    "95% interval coverage %": float(np.mean((actual_test >= test_ci95.iloc[:, 0]) & (actual_test <= test_ci95.iloc[:, 1])) * 100),
}
for name, value in accuracy.items():
    print(f"{name:36s}: {value:,.2f}")

MAPE %                              : 3.34
MAE (trips per workday)             : 13,832.05
RMSE (trips per workday)            : 15,276.45
MASE                                : 0.86
bias % (mean error / mean actual)   : 3.08
80% interval coverage %             : 87.50
95% interval coverage %             : 100.00


In [21]:
tail = train.loc["2015-01":]
history_tail = pd.Series(np.exp(tail.to_numpy()), index=tail.index)
fig = go.Figure()
fig.add_trace(go.Scatter(x=test_ci95.index, y=test_ci95.iloc[:, 1], mode="lines", line=dict(width=0),
                         showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=test_ci95.index, y=test_ci95.iloc[:, 0], mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor="rgba(42, 120, 214, 0.12)", name="95% interval", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=test_ci80.index, y=test_ci80.iloc[:, 1], mode="lines", line=dict(width=0),
                         showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=test_ci80.index, y=test_ci80.iloc[:, 0], mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor="rgba(42, 120, 214, 0.25)", name="80% interval", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=history_tail.index, y=history_tail.values, mode="lines", name="training data",
                         line=dict(color=viz.MUTED, width=2), hovertemplate="%{x|%b %Y}: %{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=test_mean.index, y=test_mean.values, mode="lines", name="forecast",
                         line=dict(color=viz.ACCENT, width=2.5, dash="dash"),
                         hovertemplate="%{x|%b %Y}: forecast %{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=actual_test.index, y=actual_test.values, mode="lines+markers", name="actual (unseen)",
                         line=dict(color=viz.INK, width=2.5), marker=dict(size=7),
                         hovertemplate="%{x|%b %Y}: actual %{y:,.0f}<extra></extra>"))
viz.vline(fig, pd.Timestamp("2018-03-01"), "model has not seen data after this line")
fig.update_yaxes(title_text="trips per workday", tickformat="~s")
fig.update_layout(hovermode="x unified")
viz.titled(fig, f"Two years ahead without seeing the data: {accuracy['MAPE %']:.1f}% average error",
           f"{CHOSEN_NAME} trained to Feb 2018, forecast Mar 2018 - Feb 2020; "
           f"{accuracy['95% interval coverage %']:.0f}% of months inside the 95% interval")
viz.save(fig, "13_holdout_forecast")
fig

บน 24 เดือนที่โมเดลไม่เคยเห็น
- MAPE 3.3%: คลาดเฉลี่ยราว 13,800 คนต่อวันทำงาน จากระดับราว 410,000
- MASE 0.86: ต่ำกว่า 1 คือแม่นกว่า seasonal naive เมื่อเทียบในสเกลของ train
- ช่วง 95% ครอบค่าจริงทั้ง 24 เดือน และช่วง 80% ครอบ 21 ใน 24 เดือน (87.5%) ใกล้ 80% ตามที่ควร แปลว่าช่วงพยากรณ์ calibrated พอดี ไม่แคบเกิน

ข้อควรระวัง: bias +3.1% ค่าจริงสูงกว่าพยากรณ์เกือบทุกเดือน
โมเดลต่อแนวโน้มลดลงของปี 2017 ออกไป แต่จริงๆ ปี 2018-2019 ผู้โดยสารเกือบนิ่ง
ถ้า bias แบบเดียวกันเกิดในช่วง 2020-2023 ด้วย counterfactual จะต่ำไปเล็กน้อย และการฟื้นตัวจริงจะต่ำกว่าค่าที่คำนวณได้ ส่วนที่ 14 ตรวจเรื่องนี้ด้วยโมเดลที่ไม่ต่อแนวโน้ม

## 12. สมการของโมเดล

fit โมเดลที่เลือกใหม่บนข้อมูลก่อน COVID ทั้งหมด (ม.ค. 2011 ถึง ก.พ. 2020) เพื่อใช้พยากรณ์จริง แล้วตรวจพารามิเตอร์และ residual ซ้ำ
สมการเขียนด้วย backshift operator B (B·yₜ = yₜ₋₁) โดย yₜ = log ของผู้โดยสารเฉลี่ยต่อวันทำงานเดือน t

In [22]:
final_fit = fit_sarima(log_pre, ORDER, SEASONAL)
params = final_fit.params


def term(value, power):
    if value >= 0:
        sign = "+"
    else:
        sign = "-"
    if power == 1:
        return f" {sign} {abs(value):.4f}B"
    return f" {sign} {abs(value):.4f}B^{{{power}}}"


left = []
right = []
for lag in range(1, ORDER[0] + 1):
    left.append(("ar", lag, float(params[f"ar.L{lag}"])))
for lag in range(1, SEASONAL[0] + 1):
    left.append(("sar", lag * 12, float(params[f"ar.S.L{lag * 12}"])))
for lag in range(1, ORDER[2] + 1):
    right.append(("ma", lag, float(params[f"ma.L{lag}"])))
for lag in range(1, SEASONAL[2] + 1):
    right.append(("sma", lag * 12, float(params[f"ma.S.L{lag * 12}"])))

left_text = ""
for kind, power, value in left:
    left_text += "(1" + term(-value, power) + ")"
right_text = ""
for kind, power, value in right:
    right_text += "(1" + term(value, power) + ")"
if right_text == "":
    right_text = "1"
differencing = ""
if ORDER[1] == 1:
    differencing += "(1 - B)"
if SEASONAL[1] == 1:
    differencing += "(1 - B^{12})"
sigma = float(np.sqrt(params["sigma2"]))
equation = (f"$$ {left_text}{differencing}\\, y_t = {right_text}\\, \\varepsilon_t, "
            f"\\qquad \\varepsilon_t \\sim N(0,\\ {sigma:.4f}^2) $$")
display(Markdown(f"**{CHOSEN_NAME}** fitted on Jan 2011 - Feb 2020:"))
display(Markdown(equation))

final_rows = []
final_conf = final_fit.conf_int()
for name in params.index:
    final_rows.append({"parameter": name, "estimate": round(float(params[name]), 4),
                       "std error": round(float(final_fit.bse[name]), 4), "p-value": round(float(final_fit.pvalues[name]), 4),
                       "95% CI": f"[{final_conf.loc[name].iloc[0]:.4f}, {final_conf.loc[name].iloc[1]:.4f}]"})
final_burn = ORDER[1] + SEASONAL[1] * 12
final_resid = np.asarray(final_fit.resid)[final_burn:]
print(f"full-sample fit: AICc {final_fit.aicc:.2f}, residual sd {np.std(final_resid):.4f} (≈ {np.std(final_resid) * 100:.1f}% per month)")
print(f"Ljung-Box lag 24 p = {acorr_ljungbox(final_resid, lags=[24], model_df=len(params) - 1)['lb_pvalue'].iloc[0]:.3f}, "
      f"Jarque-Bera p = {stats.jarque_bera(final_resid)[1]:.3f}, ARCH p = {het_arch(final_resid, nlags=12)[1]:.3f}, "
      f"smallest |root| = {np.min(np.abs(np.r_[final_fit.arroots, final_fit.maroots])):.3f}")
pd.DataFrame(final_rows).set_index("parameter")

**SARIMA(0, 1, 1)(1, 1, 0)12** fitted on Jan 2011 - Feb 2020:

$$ (1 + 0.5902B^{12})(1 - B)(1 - B^{12})\, y_t = (1 - 0.4257B)\, \varepsilon_t, \qquad \varepsilon_t \sim N(0,\ 0.0173^2) $$

full-sample fit: AICc -500.01, residual sd 0.0178 (≈ 1.8% per month)
Ljung-Box lag 24 p = 0.319, Jarque-Bera p = 0.001, ARCH p = 0.819, smallest |root| = 1.045


,estimate,std error,p-value,95% CI
parameter,,,,
ma.L1,-0.4257,0.0960,0.0,"[-0.6138, -0.2376]"
ar.S.L12,-0.5902,0.0795,0.0,"[-0.7461, -0.4344]"
sigma2,0.0003,0.0000,0.0,"[0.0002, 0.0004]"


อ่านสมการ
- (1 - B)(1 - B¹²) yₜ คือการเปลี่ยนแปลงรายเดือนของการเปลี่ยนแปลงรายปี (d=1, D=1)
- (1 + 0.5902B¹²) คือส่วน seasonal AR
- (1 - 0.4257B) εₜ คือส่วน MA ของความคลาดเคลื่อน

เมื่อ fit บนข้อมูลก่อน COVID ทั้งหมด พารามิเตอร์ใกล้เคียงกับบน train (θ₁ -0.37 เป็น -0.43, Φ₁ -0.63 เป็น -0.59) แปลว่าโมเดลเสถียร
ทั้งสองตัวยังมีนัยสำคัญ (p < 0.001) และรากยังอยู่นอกวงกลม (|root| ต่ำสุด 1.045)

residual sd 1.8% ต่อเดือน Ljung-Box และ ARCH ยังผ่าน แต่ Jarque-Bera ไม่ผ่าน (p = 0.001) ส่วนถัดไปหาสาเหตุ

### Outlier และความทนทานของผล

ตรวจ residual ของโมเดลที่ fit บนข้อมูลเต็มช่วง เดือนที่ |z| > 3 ถือว่าเป็น outlier แล้วทดสอบ 2 อย่าง
- ถ้าตัดเดือนเหล่านั้นออก residual ปกติไหม (ดูว่าปัญหา normality มาจากไม่กี่เดือนหรือทั้งชุด)
- fit ใหม่โดยใส่ intervention dummy (ตัวแปร 0/1 ของเดือนนั้น) แล้วดูว่าค่าพยากรณ์ counterfactual เปลี่ยนไหม

In [23]:
final_residuals = pd.Series(final_resid, index=log_pre.index[final_burn:])
final_z = final_residuals / final_residuals.std()
outlier_months = list(final_z[final_z.abs() > 3].index)
outlier_rows = []
for month in outlier_months:
    outlier_rows.append({"month": f"{month:%b %Y}", "z": round(float(final_z[month]), 2),
                         "actual": round(float(pre[month])),
                         "fitted": round(float(np.exp(log_pre[month] - final_residuals[month])))})
print(f"months with |z| > 3: {len(outlier_months)}")
print(pd.DataFrame(outlier_rows).to_string(index=False))
kept = final_residuals[final_z.abs() <= 3]
print(f"Jarque-Bera p: all months {stats.jarque_bera(final_residuals)[1]:.4f}, "
      f"without the outliers {stats.jarque_bera(kept)[1]:.4f}")

dummies = pd.DataFrame(index=log_pre.index)
for month in outlier_months:
    column = np.zeros(len(log_pre))
    column[list(log_pre.index).index(month)] = 1.0
    dummies[f"pulse {month:%Y-%m}"] = column
dummy_fit = fit_sarima(log_pre, ORDER, SEASONAL, exog=dummies)
dummy_resid = np.asarray(dummy_fit.resid)[final_burn:]
future_dummies = pd.DataFrame(0.0, index=post.index, columns=dummies.columns)
dummy_cf = np.exp(np.asarray(dummy_fit.get_forecast(len(post), exog=future_dummies).predicted_mean))
plain_cf = np.exp(np.asarray(final_fit.get_forecast(len(post)).predicted_mean))
print(f"with intervention dummies: Jarque-Bera p = {stats.jarque_bera(dummy_resid)[1]:.4f}, "
      f"Ljung-Box lag 24 p = {acorr_ljungbox(dummy_resid, lags=[24], model_df=len(dummy_fit.params) - 1)['lb_pvalue'].iloc[0]:.3f}")
print(f"largest change in the counterfactual: {np.max(np.abs(dummy_cf / plain_cf - 1)) * 100:.2f}%")

months with |z| > 3: 2
   month     z  actual  fitted
Oct 2013 -3.18  395043  418147
Nov 2018 -3.50  397347  423016
Jarque-Bera p: all months 0.0008, without the outliers 0.2487


with intervention dummies: Jarque-Bera p = 0.4094, Ljung-Box lag 24 p = 0.212
largest change in the counterfactual: 2.59%


ปัญหา normality มาจาก 2 เดือนเท่านั้น ถ้าตัดสองเดือนนี้ออก Jarque-Bera ผ่าน (p = 0.25)

ทั้งสองเดือนมีผู้โดยสารต่ำกว่าที่โมเดลคาดราว 6%
- ต.ค. 2013: เป็นเดือนที่มี BART strike วัน strike 2 วันถูกตัดออกจากข้อมูลแล้ว แต่วันรอบๆ อาจยังได้รับผลกระทบ
- พ.ย. 2018: มีวันที่ต่ำผิดปกติหลายวัน
  - 12 พ.ย. (305,019) เป็นวันหยุดชดเชย Veterans Day ที่ไม่ได้ถูกติดป้ายว่าเป็นวันหยุดในข้อมูล
  - 16-21 พ.ย. (286,000-358,000) ตรงกับช่วงที่ควันไฟป่า Camp Fire ปกคลุม Bay Area
  - เป็นความสอดคล้องของช่วงเวลา ยังไม่ได้พิสูจน์ว่าเป็นสาเหตุ

เมื่อใส่ intervention dummy ของสองเดือนนี้
- residual ปกติ: Jarque-Bera p = 0.41
- Ljung-Box ยังผ่าน
- counterfactual เปลี่ยนไม่เกิน 2.6% ในเดือนใดเดือนหนึ่ง และค่าเฉลี่ย ม.ค.-เม.ย. 2023 แทบไม่เปลี่ยน (ดูส่วนที่ 14)

สรุปคือ outlier ไม่ได้บิดผลลัพธ์ จึงใช้โมเดลที่ไม่มี dummy เป็นหลัก เพราะเรียบง่ายกว่า และไม่ต้องตัดสินใจเองว่าเดือนไหนผิดปกติ

## 13. Counterfactual: ถ้าไม่มี COVID

พยากรณ์ 38 เดือน (มี.ค. 2020 ถึง เม.ย. 2023) จากโมเดลที่เห็นข้อมูลถึง ก.พ. 2020 เท่านั้น
- ค่ากลางใช้ exp ของค่าพยากรณ์ใน log ซึ่งเป็น median ของการแจกแจงแบบ log-normal
- ช่วงพยากรณ์ 80% และ 95% กว้างขึ้นตามระยะเวลา เพราะความไม่แน่นอนสะสม

การฟื้นตัว = ค่าจริง ÷ counterfactual และช่วงของการฟื้นตัวคำนวณจากขอบของช่วงพยากรณ์

In [24]:
HORIZON_CF = len(post)
cf = final_fit.get_forecast(HORIZON_CF)
cf_mid = np.exp(cf.predicted_mean)
cf95 = np.exp(cf.conf_int(alpha=0.05))
cf80 = np.exp(cf.conf_int(alpha=0.20))
cf_mid.index = post.index
cf95.index = post.index
cf80.index = post.index

recovery = post / cf_mid * 100
recovery_low = post / cf95.iloc[:, 1] * 100
recovery_high = post / cf95.iloc[:, 0] * 100

same_month_2019 = []
for month in post.index:
    same_month_2019.append(trips[pd.Timestamp(year=2019, month=month.month, day=1)])
recovery_vs_2019 = post / np.array(same_month_2019) * 100

lost_per_month = (cf_mid - post) * workdays[POST_START:]
lost_total = float(lost_per_month.sum())

window = slice("2023-01", "2023-04")
cf_2023 = float(cf_mid[window].mean())
actual_2023 = float(post[window].mean())
baseline_2019 = float(trips["2019-01":"2019-04"].mean())
recovery_2023 = actual_2023 / cf_2023 * 100
recovery_2023_low = actual_2023 / float(cf95[window].iloc[:, 1].mean()) * 100
recovery_2023_high = actual_2023 / float(cf95[window].iloc[:, 0].mean()) * 100

print(f"Jan-Apr 2023 actual           : {actual_2023:,.0f} trips per workday")
print(f"Jan-Apr 2023 no-COVID forecast: {cf_2023:,.0f} (95% interval {cf95[window].iloc[:, 0].mean():,.0f} - {cf95[window].iloc[:, 1].mean():,.0f})")
print(f"Jan-Apr 2019 actual           : {baseline_2019:,.0f}")
print(f"recovery vs counterfactual    : {recovery_2023:.1f}% (95% range {recovery_2023_low:.1f}% - {recovery_2023_high:.1f}%)")
print(f"recovery vs 2019 (old method) : {actual_2023 / baseline_2019 * 100:.1f}%")
print(f"workday trips lost Mar 2020 - Apr 2023: {lost_total / 1e6:,.1f} million (normal workdays only)")
print(f"lowest month: {recovery.idxmin():%b %Y} at {recovery.min():.1f}% of the no-COVID level")

yearly_recovery = recovery.groupby(recovery.index.year).mean().round(1).to_frame("recovery vs counterfactual %")
yearly_recovery["recovery vs 2019 %"] = recovery_vs_2019.groupby(recovery_vs_2019.index.year).mean().round(1)
yearly_recovery

Jan-Apr 2023 actual           : 151,848 trips per workday
Jan-Apr 2023 no-COVID forecast: 389,620 (95% interval 319,562 - 475,069)
Jan-Apr 2019 actual           : 409,777
recovery vs counterfactual    : 39.0% (95% range 32.0% - 47.5%)
recovery vs 2019 (old method) : 37.1%
workday trips lost Mar 2020 - Apr 2023: 238.4 million (normal workdays only)
lowest month: Apr 2020 at 6.3% of the no-COVID level


,recovery vs counterfactual %,recovery vs 2019 %
month,,
2020,14.0,13.7
2021,20.1,19.6
2022,34.3,33.0
2023,38.9,37.0


### ภาพหลัก: ช่องว่างระหว่างโลกที่มี COVID กับโลกที่ไม่มี

In [25]:
LOST_FILL = "rgba(227, 73, 72, 0.22)"
fig = go.Figure()
fig.add_trace(go.Scatter(x=cf95.index, y=cf95.iloc[:, 1], mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=cf95.index, y=cf95.iloc[:, 0], mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor="rgba(42, 120, 214, 0.10)", name="no-COVID 95% interval", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=cf80.index, y=cf80.iloc[:, 1], mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=cf80.index, y=cf80.iloc[:, 0], mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor="rgba(42, 120, 214, 0.20)", name="no-COVID 80% interval", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=cf_mid.index, y=cf_mid.values, mode="lines", name="no-COVID forecast (SARIMA)",
                         line=dict(color=viz.ACCENT, width=3, dash="dash"),
                         hovertemplate="%{x|%b %Y}: expected %{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=post.index, y=post.values, mode="lines", name="actual after COVID",
                         line=dict(color=viz.SERIES[7], width=3), fill="tonexty", fillcolor=LOST_FILL,
                         customdata=recovery.values,
                         hovertemplate="%{x|%b %Y}: actual %{y:,.0f} (%{customdata:.0f}% of expected)<extra></extra>"))
fig.add_trace(go.Scatter(x=pre.index, y=pre.values, mode="lines", name="actual before COVID",
                         line=dict(color=viz.INK, width=2.5), hovertemplate="%{x|%b %Y}: %{y:,.0f}<extra></extra>"))
viz.vline(fig, pd.Timestamp("2020-03-01"), "shelter in place")
fig.add_annotation(x=pd.Timestamp("2021-09-01"), y=float(cf_mid["2021-09"].iloc[0]) * 0.62,
                   text=(f"<b>{lost_total / 1e6:,.0f} million</b><br>workday trips that<br>never happened"
                         f"<br><span style='font-size:14px'>Apr 2023: {recovery.iloc[-1]:.0f}% of the no-COVID level</span>"),
                   showarrow=False, font=dict(size=18, color=viz.SERIES[7]), align="center")
fig.update_yaxes(title_text="average trips per workday", tickformat="~s", rangemode="tozero")
fig.update_layout(hovermode="x unified", legend=dict(y=1.12), margin=dict(t=170))
viz.titled(fig, f"Without COVID, BART would carry ~{cf_2023 / 1000:,.0f}k trips per workday in early 2023; it carries {actual_2023 / 1000:,.0f}k",
           f"Monthly trips per normal workday; dashed line and bands = SARIMA forecast from data up to Feb 2020; "
           f"red area = {lost_total / 1e6:,.0f} million workday trips lost")
viz.save(fig, "13_counterfactual_gap", width=1400, height=700)
fig

### ภาพเคลื่อนไหว: ช่องว่างค่อยๆ เปิดทีละเดือน

กด Play เพื่อดูตั้งแต่ มี.ค. 2020 ถึง เม.ย. 2023 ตัวเลขด้านขวาคือจำนวนทริปที่หายไปสะสม และ % การฟื้นตัวของเดือนนั้น

In [26]:
context = trips.loc["2018-01":PRE_END]
anim_months = list(post.index)


def gap_traces(upto):
    shown_post = post[:upto]
    shown_cf = cf_mid[:upto]
    return [
        go.Scatter(x=shown_cf.index, y=shown_cf.values, mode="lines", line=dict(color=viz.ACCENT, width=3, dash="dash"),
                   name="no-COVID forecast", hovertemplate="%{x|%b %Y}: expected %{y:,.0f}<extra></extra>"),
        go.Scatter(x=shown_post.index, y=shown_post.values, mode="lines+markers", line=dict(color=viz.SERIES[7], width=3),
                   marker=dict(size=6), fill="tonexty", fillcolor=LOST_FILL, name="actual",
                   hovertemplate="%{x|%b %Y}: actual %{y:,.0f}<extra></extra>"),
    ]


def gap_notes(upto):
    lost_so_far = float(lost_per_month[:upto].sum())
    month_recovery = float(recovery[upto])
    return [
        dict(text=f"<b>{upto:%b %Y}</b>", x=0.01, y=0.58, xref="paper", yref="paper", xanchor="left", yanchor="top",
             showarrow=False, font=dict(size=40, color=viz.CONTEXT)),
        dict(text=f"<b>{lost_so_far / 1e6:,.1f} M</b> workday trips lost so far", x=0.01, y=0.40, xref="paper",
             yref="paper", xanchor="left", yanchor="top", showarrow=False, font=dict(size=18, color=viz.SERIES[7])),
        dict(text=f"this month: <b>{month_recovery:.0f}%</b> of the no-COVID level", x=0.01, y=0.31, xref="paper",
             yref="paper", xanchor="left", yanchor="top", showarrow=False, font=dict(size=15, color=viz.INK2)),
    ]


fig = go.Figure()
fig.add_trace(go.Scatter(x=cf95.index, y=cf95.iloc[:, 1], mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=cf95.index, y=cf95.iloc[:, 0], mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor="rgba(42, 120, 214, 0.10)", name="95% interval", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=context.index, y=context.values, mode="lines", line=dict(color=viz.INK, width=2.5),
                         name="before COVID", hovertemplate="%{x|%b %Y}: %{y:,.0f}<extra></extra>"))
for trace in gap_traces(anim_months[-1]):
    fig.add_trace(trace)

frames = []
steps = []
for month in anim_months:
    name = f"{month:%b %Y}"
    frames.append(go.Frame(name=name, data=gap_traces(month), traces=[3, 4], layout=dict(annotations=gap_notes(month))))
    steps.append(dict(label=name, method="animate",
                      args=[[name], dict(mode="immediate", frame=dict(duration=0, redraw=False), transition=dict(duration=0))]))
fig.frames = frames
play = dict(label="Play", method="animate",
            args=[None, dict(frame=dict(duration=260, redraw=False), fromcurrent=True, transition=dict(duration=120))])
pause = dict(label="Pause", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate", transition=dict(duration=0))])
fig.update_layout(
    annotations=gap_notes(anim_months[-1]),
    updatemenus=[dict(type="buttons", buttons=[play, pause], direction="left", x=0, y=-0.16, xanchor="left",
                      yanchor="top", showactive=False)],
    sliders=[dict(steps=steps, active=len(steps) - 1, x=0.16, len=0.84, y=-0.13, currentvalue=dict(visible=False),
                  pad=dict(t=0))],
    hovermode="x unified", margin=dict(t=150, b=150),
)
fig.update_xaxes(range=[pd.Timestamp("2018-01-01"), pd.Timestamp("2023-06-01")])
fig.update_yaxes(title_text="average trips per workday", tickformat="~s", range=[0, float(cf95.iloc[:, 1].max()) * 1.05])
viz.titled(fig, "Month by month, the gap between the world with COVID and the world without it",
           "Press Play: red area = workday trips lost against the SARIMA no-COVID forecast")
viz.save(fig, "13_gap_animation", width=1300, height=680)
fig

### การฟื้นตัวรายเดือน: เทียบ counterfactual กับวิธีเดิม (เทียบปี 2019)

In [27]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=recovery.index, y=recovery_low.values, mode="lines", line=dict(width=0), showlegend=False,
                         hoverinfo="skip"))
fig.add_trace(go.Scatter(x=recovery.index, y=recovery_high.values, mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor="rgba(42, 120, 214, 0.18)", name="range from the 95% forecast interval", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=recovery.index, y=recovery.values, mode="lines+markers", name="vs no-COVID forecast (new)",
                         line=dict(color=viz.ACCENT, width=3), marker=dict(size=6),
                         hovertemplate="%{x|%b %Y}: %{y:.1f}%<extra>vs counterfactual</extra>"))
fig.add_trace(go.Scatter(x=recovery_vs_2019.index, y=recovery_vs_2019.values, mode="lines", name="vs same month of 2019 (old)",
                         line=dict(color=viz.MUTED, width=2, dash="dot"),
                         hovertemplate="%{x|%b %Y}: %{y:.1f}%<extra>vs 2019</extra>"))
fig.add_hline(y=100, line=dict(color=viz.INK2, dash="dash", width=1))
fig.add_annotation(x=recovery.index[2], y=100, text="full recovery", showarrow=False, yshift=12,
                   font=dict(size=11, color=viz.INK2))
viz.end_label(fig, recovery.index[-1], float(recovery.iloc[-1]), f"{recovery.iloc[-1]:.0f}%", color=viz.ACCENT, ax=-40, ay=-40)
fig.update_yaxes(title_text="actual as % of the reference", ticksuffix="%", range=[0, 110])
fig.update_layout(hovermode="x unified")
viz.titled(fig, f"Jan-Apr 2023 ridership is {recovery_2023:.0f}% of the no-COVID level ({recovery_2023_low:.0f}-{recovery_2023_high:.0f}% with forecast uncertainty)",
           "Monthly recovery measured two ways: against the SARIMA counterfactual and against the same month of 2019")
viz.save(fig, "13_recovery_curve")
fig

คำตอบ Q1 ฉบับนักสถิติ
- ม.ค.-เม.ย. 2023 ผู้โดยสารจริง 151,848 คนต่อวันทำงาน
- ถ้าไม่มี COVID โมเดลคาดว่าจะเป็น 389,620 คน (ช่วง 95%: 319,562 ถึง 475,069)
- การฟื้นตัวจึงเท่ากับ 39.0% ของระดับที่ควรเป็น ช่วงความไม่แน่นอนอยู่ที่ 32.0-47.5%
- วิธีเดิมที่เทียบกับปี 2019 ได้ 37.1% ต่างกันราว 2 จุด เพราะโมเดลคาดว่าถ้าไม่มี COVID ผู้โดยสารจะลดลงต่อเล็กน้อย (2023 ต่ำกว่า 2019 ราว 5%)

เส้นทางการฟื้นตัว
- ต่ำสุดเดือน เม.ย. 2020 ที่ 6.3%
- ค่าเฉลี่ยรายปี: 2020 อยู่ที่ 14%, 2021 อยู่ที่ 20%, 2022 อยู่ที่ 34% และ ม.ค.-เม.ย. 2023 อยู่ที่ 39%
- ตั้งแต่ ก.ย. 2022 การฟื้นตัวทรงตัวที่ราว 36-41% ไม่ได้ขึ้นต่อชัดเจน
- จุดแรก (มี.ค. 2020) สูงเพราะ shelter in place เริ่มกลางเดือน

ทริปที่หายไปตั้งแต่ มี.ค. 2020 ถึง เม.ย. 2023 รวม 238 ล้านทริป
ตัวเลขนี้นับเฉพาะวันทำงานปกติ ไม่รวมเสาร์อาทิตย์และวันหยุด ยอดจริงจึงสูงกว่านี้

## 14. ความไวต่อการเลือกโมเดล

ถ้าเลือกโมเดลอื่นใน shortlist ผลการฟื้นตัวจะเปลี่ยนไปแค่ไหน ถ้าทุกโมเดลให้คำตอบใกล้กัน แปลว่าข้อสรุปไม่ได้ขึ้นกับการเลือกโมเดล

In [28]:
sensitivity_rows = []
for order, seasonal_order in SHORTLIST:
    result = fit_sarima(log_pre, order, seasonal_order)
    forecast = np.exp(np.asarray(result.get_forecast(HORIZON_CF).predicted_mean))
    forecast_series = pd.Series(forecast, index=post.index)
    expected = float(forecast_series[window].mean())
    lost = float(((forecast_series - post) * workdays[POST_START:]).sum())
    sensitivity_rows.append({"model": label(order, seasonal_order), "chosen": flag(label(order, seasonal_order) == CHOSEN_NAME),
                             "no-COVID Jan-Apr 2023": round(expected), "recovery %": round(actual_2023 / expected * 100, 1),
                             "trips lost (million)": round(lost / 1e6, 1)})
alternative_cf = pd.Series(np.exp(np.asarray(alternative_fit.get_forecast(HORIZON_CF).predicted_mean)), index=post.index)
sensitivity_rows.append({"model": label(ALTERNATIVE[0], ALTERNATIVE[1]) + " (no d)", "chosen": "",
                         "no-COVID Jan-Apr 2023": round(float(alternative_cf[window].mean())),
                         "recovery %": round(actual_2023 / float(alternative_cf[window].mean()) * 100, 1),
                         "trips lost (million)": round(float(((alternative_cf - post) * workdays[POST_START:]).sum()) / 1e6, 1)})
dummy_series = pd.Series(dummy_cf, index=post.index)
sensitivity_rows.append({"model": CHOSEN_NAME + " + outlier dummies", "chosen": "",
                         "no-COVID Jan-Apr 2023": round(float(dummy_series[window].mean())),
                         "recovery %": round(actual_2023 / float(dummy_series[window].mean()) * 100, 1),
                         "trips lost (million)": round(float(((dummy_series - post) * workdays[POST_START:]).sum()) / 1e6, 1)})
ets_full = ExponentialSmoothing(pre, trend="add", damped_trend=True, seasonal="mul", seasonal_periods=12).fit()
ets_cf = pd.Series(np.asarray(ets_full.forecast(HORIZON_CF)), index=post.index)
sensitivity_rows.append({"model": "ETS (damped, mult.)", "chosen": "", "no-COVID Jan-Apr 2023": round(float(ets_cf[window].mean())),
                         "recovery %": round(actual_2023 / float(ets_cf[window].mean()) * 100, 1),
                         "trips lost (million)": round(float(((ets_cf - post) * workdays[POST_START:]).sum()) / 1e6, 1)})
sensitivity_rows.append({"model": "flat at 2019 level (old method)", "chosen": "", "no-COVID Jan-Apr 2023": round(baseline_2019),
                         "recovery %": round(actual_2023 / baseline_2019 * 100, 1), "trips lost (million)": None})
sensitivity = pd.DataFrame(sensitivity_rows).set_index("model")
sensitivity.to_csv(TAB / "q1_counterfactual_sensitivity.csv")
sensitivity

,chosen,no-COVID Jan-Apr 2023,recovery %,trips lost (million)
model,,,,
"SARIMA(0, 1, 1)(1, 1, 1)12",✗,394201,38.5,240.1
"SARIMA(0, 1, 1)(0, 1, 1)12",✗,406200,37.4,245.3
"SARIMA(0, 1, 1)(0, 1, 2)12",✗,389782,39.0,238.2
"SARIMA(0, 1, 1)(1, 1, 0)12",✓,389620,39.0,238.4
"SARIMA(2, 1, 0)(0, 1, 2)12",✗,383648,39.6,235.1
"SARIMA(1, 0, 0)(0, 1, 1)12 (no d)",,403212,37.7,244.2
"SARIMA(0, 1, 1)(1, 1, 0)12 + outlier dummies",,389291,39.0,238.6
"ETS (damped, mult.)",,400410,37.9,242.4
flat at 2019 level (old method),,409777,37.1,NaN


ทุกโมเดลให้คำตอบในช่วงแคบ: การฟื้นตัว 37.1% ถึง 39.6% และทริปที่หายไป 235 ถึง 245 ล้านทริป ครอบคลุมทุกแบบที่ทดสอบ
- SARIMA 5 แบบ
- โมเดลที่ไม่ต่อแนวโน้ม (no d)
- โมเดลที่มี outlier dummy
- ETS
- เส้นแบนที่ระดับปี 2019

แม้ความแม่นระยะยาวของแต่ละโมเดลมีขีดจำกัด (ส่วนที่ 7) คำตอบหลักไม่ขึ้นกับการเลือกโมเดล
ข้อสรุปที่มั่นใจได้คือ ต้นปี 2023 BART มีผู้โดยสารวันทำงานไม่ถึง 2 ใน 5 ของระดับที่ควรเป็นถ้าไม่มี COVID

## 15. สรุป

### Model card

| หัวข้อ | ผล |
|---|---|
| โมเดล | SARIMA(0,1,1)(1,1,0)12 บน log ผู้โดยสารเฉลี่ยต่อวันทำงาน |
| วิธีระบุ | ADF + KPSS ได้ d=1, D=1 แล้ว ACF/PACF มี spike ที่ lag 1 และ 12 จึงค้นหา 117 โมเดล |
| พารามิเตอร์ | θ₁ = -0.426, Φ₁ = -0.590 ทั้งคู่ p < 0.001 รากอยู่นอกวงกลม |
| residual | ผ่าน Ljung-Box, ARCH; ความปกติผ่านเมื่อจัดการ outlier 2 เดือน |
| ความแม่น 24 เดือนที่ไม่เคยเห็น | MAPE 3.3%, MASE 0.86, ช่วง 95% ครอบค่าจริง 24 ใน 24 เดือน |
| ความแม่น rolling CV | 1 เดือน 1.4%, 12 เดือน 4.5%, 24 เดือน 7.6% |
| ข้อจำกัด | ระยะเกิน 4 เดือน ไม่แม่นกว่า seasonal naive อย่างมีนัยสำคัญ |
| ความไวต่อโมเดล | การฟื้นตัว 37.1-39.6% ในทุก 9 แบบ |

### Insight และ Pain point

| Insight | Pain point สำหรับ BART |
|---|---|
| ต้นปี 2023 ผู้โดยสารวันทำงานอยู่ที่ 39% ของระดับที่ควรเป็นถ้าไม่มี COVID (ช่วง 32-48%) | รายได้ค่าโดยสารยังขาดมากกว่าครึ่ง ต้องพึ่งเงินอุดหนุนต่อ งบประมาณควรวางบนระดับปัจจุบัน ไม่ใช่สมมติว่าจะกลับไป 2019 |
| ตัวเลขแทบเท่าวิธีเดิมที่เทียบกับปี 2019 (37%) เพราะก่อน COVID ผู้โดยสารลดลงอยู่แล้วตั้งแต่ 2016 | ปัญหาไม่ได้เริ่มที่ COVID เป้าหมาย "กลับไปเท่า 2019" อาจไม่ใช่เส้นที่ถูก ต้องแก้ทั้งผลของ COVID และแนวโน้มลดลงเดิม |
| การฟื้นตัวทรงตัวที่ 36-41% ตั้งแต่ ก.ย. 2022 | ไม่ควรรอให้ผู้โดยสารกลับมาเอง ต้องมีมาตรการดึงผู้โดยสารเชิงรุก |
| ทริปที่หายไปสะสม 238 ล้านทริป นับเฉพาะวันทำงานปกติ | ความเสียหายสะสมใหญ่กว่าที่เห็นจากตัวเลขรายเดือน ใช้เป็นตัวเลขอ้างอิงตอนขอเงินสนับสนุน |
| โมเดลพยากรณ์ 1-4 เดือนแม่นราว 1-2% แต่ระยะ 2 ปีขึ้นไปคลาดได้ 4-8% | ใช้ SARIMA ช่วยวางแผนรายไตรมาสได้ดี แต่แผนระยะยาวต้องใช้ scenario และช่วง ไม่ใช่ตัวเลขเดียว |
| เดือนที่โมเดลพลาดมากตรงกับเหตุการณ์ภายนอก (strike ปี 2013, ช่วงควันไฟป่าปลายปี 2018) | เหตุการณ์ภายนอกทำให้ผู้โดยสารลดลงได้ราว 6% ในเดือนนั้น ควรมีแผนรับมือ เช่น สื่อสารล่วงหน้าและปรับความถี่รถ |

### ตอบอาจารย์: ทำไมเชื่อตัวเลขนี้ได้
- ทำครบทุกขั้นของ Box-Jenkins
  - การแปลงมีเหตุผลรองรับ
  - ทดสอบ stationarity สองทาง
  - ระบุโมเดลจาก ACF/PACF แล้วค้นหาจริง 117 แบบ
- เลือกโมเดลด้วยกฎที่เขียนไว้ก่อนดูผล counterfactual
- ตรวจพารามิเตอร์ทุกตัวด้วย z-test, LR test และ correlation ของตัวประมาณ
- ตรวจ assumption ด้วย test 9 แบบ และอธิบายจุดที่ไม่ผ่าน
- วัดความแม่นทั้งบน holdout และ rolling CV เทียบกับ benchmark พร้อม Diebold-Mariano
- รายงานข้อจำกัดระยะยาวตรงๆ และพิสูจน์ว่าข้อสรุปไม่ขึ้นกับการเลือกโมเดล